[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/conformal_circle/conformal_circle.ipynb)

# A Circle Constructed Twice

This example constructs a single horizontal circle in conformal space in two dual ways, reflects both constructions in the circle's own plane, and compares rigid rotation with a conformal [hyperbolic glide reflection](https://hamishtodd1.substack.com/p/hyperbolic-glide-reflections) along the same circle. Joining three ordered points on a circle gives the curve a circulation direction along its rim, whereas intersecting an oriented plane and an oriented sphere gives the curve the transverse sense of its two cutting surfaces, so reflecting in the circle's plane leaves the point order untouched while flipping the plane's normal.

In three-dimensional conformal geometric algebra, points are modelled as null intrinsic antivectors and spheres and planes as extrinsic vectors. Joining three ordered points via `points[0] & points[1] & points[2]` forms an intrinsic `Path`, wedging two surfaces via `plane ^ sphere` forms an extrinsic `Circle`, and reflecting either via `mirror >> x` tracks both orientations automatically without sign rules, while contracting `point_join.dual()` with the horizon sphere or a mirror plane extracts the extrinsic bivector generators of Euclidean rotation and conformal boosts along the circle.

In vector-based conformal geometric algebra where points are null vectors $P_i \in \mathbb{R}^{4,1}$, the point-join circle is the outer product $C = P_1 \wedge P_2 \wedge P_3$ and the surface-meet circle is the dual $(\pi \wedge \sigma)^* = (\pi \wedge \sigma) I_5^{-1}$, so reflecting a grade-$k$ blade via $(-1)^k M X M^{-1}$ requires separate sign conventions for direct blades versus dual blades because $M I_5 M^{-1} = -I_5$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import CGA3D
from examples.util.animation import save_figure, show_states
from examples.orientation.conformal_circle import render, scenarios

np.set_printoptions(precision=3, suppress=True)

# 3D conformal space `x+y+z+e+w-`, with `+` extrinsic and `-` intrinsic.
ga = CGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
Point = ga.gatype.antivector()                                                  # Antivector-
Sphere = ga.gatype.vector()                                                     # Vector+
Circle = ga.gatype.bivector()                                                   # Bivector+
Path = ga.gatype.antitrivector()                                                # Antitrivector-

## Point joins and surface intersections under reflection

The unit sphere meets the horizontal plane in the unit circle passing through the three ordered points. Reflecting in that horizontal plane fixes every point on the circle and fixes the sphere, while reversing the plane's chosen side.

In [ ]:
points = stack([mv.x.dual() + mv.e.dual(), -mv.y.dual() + mv.e.dual(),
                -mv.x.dual() + mv.e.dual()])                                    # [3] Point-
plane = mv.z                                                                    # [] Sphere+
sphere = mv.w                                                                   # [] Sphere+
mirror = plane                                                                  # [] Sphere+
n_samples = scenarios.n_samples
n_meridians = scenarios.n_meridians
n_parallels = scenarios.n_parallels
angles = np.linspace(0, 2 * np.pi, n_samples, endpoint=False)
longitudes = np.linspace(0, 2 * np.pi, n_meridians, endpoint=False)
latitudes = np.linspace(-np.pi / 2, np.pi / 2, n_parallels)

In [ ]:
def circle_constructions(points: Point, plane: Sphere, sphere: Sphere) -> tuple[Path, Circle]:
    """Joining three ordered `points` (`[3] Point-`) gives an intrinsic circle (`[] Path-`), while
    wedging `plane` and `sphere` (`[] Sphere+`) gives an extrinsic circle (`[] Circle+`)."""
    point_join = (points[0] & points[1] & points[2]).normalized()               # [] Path-
    surface_meet = (plane ^ sphere).normalized()                                # [] Circle+
    return point_join, surface_meet


def sphere_samples(seed: Point, equator: Circle, meridian: Circle,
                   longitudes: np.ndarray, latitudes: np.ndarray) -> Point:
    """Two orthogonal rotations carry `seed` across `latitudes` (`[n_parallels]`) and `longitudes`
    (`[n_meridians]`) into a spherical grid (`[n_parallels, n_meridians] Point-`)."""
    longitude_turns = (equator * (-longitudes[None, :] / 2)).exp()              # [1, ...] Rotor+
    latitude_turns = (meridian * (-latitudes[:, None] / 2)).exp()               # [..., 1] Rotor+
    return (longitude_turns * latitude_turns) >> seed


point_join, surface_meet = circle_constructions(points, plane, sphere)          # [] Path-, Circle+
reflected_join = mirror >> point_join                                           # [] Path-
reflected_meet = mirror >> surface_meet                                         # [] Circle+

turns = (mv.xy * (angles / 2)).exp()                                            # [n_samples] Rotor+
curve = turns >> points[0]                                                      # [n_samples] Point-
surface = sphere_samples(points[0], mv.xy, mv.xz, longitudes, latitudes)        # [n_parallels, n_meridians] Point-
markers = stack([points, mirror >> points])                                     # [2, 3] Point-
curves = stack([curve, mirror >> curve])                                        # [2, n_samples] Point-
surfaces = stack([surface, mirror >> surface])                                  # [2, n_parallels, n_meridians] Point-
planes = stack([plane, mirror >> plane])                                        # [2] Sphere+
spheres = stack([sphere, mirror >> sphere])                                     # [2] Sphere+

print("Point join:\n" + point_join.coefficient_table())
print("Surface meet:\n" + surface_meet.coefficient_table())
print("Point join, reflected:\n" + reflected_join.coefficient_table())
print("Surface meet, reflected:\n" + reflected_meet.coefficient_table())
figure = render.circle_figure(markers, curves, surfaces, planes, spheres)
save_figure(figure, "orientation_conformal_circle")
figure

The left panel shows the original scene and the right panel shows its reflection in the horizontal plane. Dark chevrons along the rim and the three coloured point balls retain their circulation order, and the orange normal arrow still points into the sphere, whereas the teal normal arrow flips with the reflected plane so the intrinsic point join stays `+zw` while the extrinsic surface meet flips to `-zw`.

## Rigid rotation and conformal glide along the circle

Contracting `point_join.dual()` with the sphere at infinity selects the Euclidean rotation axis perpendicular to the disk, whereas contracting `point_join.dual()` with a vertical mirror plane selects a commuting conformal boost that crowds and stretches points along the circle between two fixed points on the mirror.

In [ ]:
origin = scenarios.origin                                                       # [] Point-
infinity = scenarios.infinity                                                   # [] Point-
horizon = scenarios.horizon                                                     # [] Sphere+
reflector = scenarios.reflector                                                 # [] Sphere+
fixed_points = scenarios.fixed_points                                           # [2] Point-
n_frames = scenarios.n_frames
n_radii = scenarios.n_radii
n_angles = scenarios.n_angles
glide_amplitude = scenarios.glide_amplitude
frame_duration_ms = scenarios.frame_duration_ms
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
radii = np.linspace(0, 1, n_radii + 1)
disk_angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)

In [ ]:
def disk_samples(direction: Point, origin: Point, infinity: Point, axis: Circle,
                 radii: np.ndarray, angles: np.ndarray) -> Point:
    """Translating `origin` along `direction` by `radii` (`[n_radii + 1]`) and turning about `axis` by
    `angles` (`[n_angles]`) samples a polar disk grid (`[n_radii + 1, n_angles] Point-`)."""
    # Dualizing the join of the direction with the point at infinity gives the translation generator along that direction.
    radial_shifts = ((infinity & direction).dual() * (-radii / 2)).exp()        # [...] Rotor+
    radial_seeds = radial_shifts >> origin                                      # [...] Point-
    angular_turns = (axis * (angles / 2)).exp()                                 # [...] Rotor+
    return angular_turns[None, :] >> radial_seeds[:, None]


def circle_motion_states(circle: Path, horizon: Sphere, reflector: Sphere, grid: Point,
                         phases: np.ndarray, glide_amplitude: float) -> Iterator[Point]:
    """Yields paired disk grids (`[2, n_radii + 1, n_angles] Point-`) carried along `circle` by rigid
    rotation and by a hyperbolic glide reflection across `phases` (`[n_frames]`)."""
    # Contracting the circle's dual with the sphere at infinity selects the Euclidean rotation generator about the circle's axis.
    axis = (circle.dual() | horizon).normalized()                               # [] Circle+
    rotations = (axis * (-phases / 2)).exp()                                    # [...] Rotor+
    # Contracting the circle's dual with the reflecting plane selects the commuting conformal boost along the circle.
    boost = reflector | circle.dual()                                           # [] Circle+
    boost_rotors = (boost * (glide_amplitude * np.sin(phases))).exp()           # [...] Rotor+
    glides = reflector * boost_rotors                                           # [...] Odd+
    for rotation, glide in zip(rotations, glides):
        yield stack([rotation >> grid, glide >> grid])


axis = (point_join.dual() | horizon).normalized()                               # [] Circle+
grid = disk_samples(mv.x.dual(), origin, infinity, axis, radii, disk_angles)    # [n_radii + 1, n_angles] Point-
motion_pairs = show_states(
    circle_motion_states(point_join, horizon, reflector, grid, phases, glide_amplitude),
    render.circle_motions_animation, fixed_points,
    name="orientation_circle_motions", duration_ms=frame_duration_ms, n_frames=n_frames,
)

Coloured rim markers track matching sample points across both disks: the left disk turns rigidly with constant spoke spacing, while the right disk reflects across the vertical diameter and surges back and forth between the two ringed fixed points under the conformal boost.

In [ ]:
# checks
rebuilt_join, rebuilt_meet = circle_constructions(markers[1], planes[1], spheres[1])
basis_v = mv.basis()                                                            # [5] Sphere+
trivectors = basis_v[:, None, None] ^ basis_v[None, :, None] ^ basis_v[None, None, :]  # [5, 5, 5] Trivector+
antibivectors = (basis_v[:, None] ^ basis_v[None, :]).dual()                    # [5, 5] Antibivector-
assert ((trivectors & (reflected_join - point_join)).abs() < 1e-12).all()
assert ((reflected_meet + surface_meet) & antibivectors).abs().max() < 1e-12
assert ((trivectors & (rebuilt_join - reflected_join)).abs() < 1e-12).all()
assert ((rebuilt_meet - reflected_meet) & antibivectors).abs().max() < 1e-12
assert ((point_join & motion_pairs[n_frames // 4][:, -1]).antinorm().abs() < 1e-12).all()